In [17]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
import os
from torchvision import transforms
from PIL import Image
import numpy as np
from tqdm import tqdm
import snntorch as snn
import time
import subprocess
import re
import thop
from snntorch import surrogate
from snntorch import spikegen
import torch.nn.functional as F
from snntorch import utils

In [2]:
DATA_DIR = "../../data"
DATASET_DIR = f"{DATA_DIR}/processed"

In [3]:
class CustomDatasetSpike(Dataset):
    def __init__(self, dataframe, image_dir, transform=None, num_steps=4):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
        self.num_steps = num_steps

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.dataframe.iloc[idx]["Image"])
        with Image.open(image_path) as img:
            if self.transform:
                img = self.transform(img)
            label = torch.as_tensor(self.dataframe.iloc[idx]["Label"], dtype=torch.long)
            img = spikegen.rate(img, num_steps=self.num_steps, gain=1)
            return img, label


class CustomDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.dataframe.iloc[idx]["Image"])
        with Image.open(image_path) as img:
            if self.transform:
                img = self.transform(img)
            label = torch.as_tensor(self.dataframe.iloc[idx]["Label"], dtype=torch.long)
            return img, label

In [4]:
test_df = pd.read_csv(f"{DATASET_DIR}/test1.csv")     
test_df_multi = pd.read_csv(f"{DATASET_DIR}/test1_multi.csv")   

In [5]:
LE = LabelEncoder()
LE_multi = LabelEncoder()
BATCH_SIZE = 40

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor()
])

LE = LabelEncoder()
LE.fit(test_df["Label"])

# LE.classes_

# swap classes in the label encoder
swapped_classes = LE.classes_.copy()
swapped_classes[0], swapped_classes[1] = swapped_classes[1], swapped_classes[0]

LE.classes_ = swapped_classes

test_df_encoded = test_df.copy()
test_df_encoded["Label"] = LE.transform(test_df_encoded["Label"])

# Multi
LE_multi.fit(test_df_multi["Label"])
test_df_multi_encoded = test_df_multi.copy()
test_df_multi_encoded["Label"] = LE_multi.transform(test_df_multi_encoded["Label"])


In [6]:
# Binary Classification
test_data_sample = test_df_encoded.groupby("Label").sample(6000, random_state=42)

# shuffle the data
test_data_sample = test_data_sample.sample(frac=1, random_state=42)
test_dataset = CustomDataset(test_data_sample, f"{DATASET_DIR}/images", transform=transform)
test_dataset_spike = CustomDatasetSpike(test_data_sample, f"{DATASET_DIR}/images", transform=transform)


# Multi Classification
test_data_sample_multi = test_df_multi_encoded.groupby("Label").sample(2000, random_state=42)

# shuffle the data
test_data_sample_multi = test_data_sample_multi.sample(frac=1, random_state=42)
test_dataset_multi = CustomDataset(test_data_sample_multi, f"{DATASET_DIR}/images", transform=transform)

In [7]:
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader_spike = DataLoader(test_dataset_spike, batch_size=BATCH_SIZE, shuffle=False)
test_loader_multi = DataLoader(test_dataset_multi, batch_size=BATCH_SIZE, shuffle=False)

In [25]:
import torch
import time
import re
import subprocess
import threading
from tqdm import tqdm
import thop

def calculate_metrics(model_class, weight_path, test_dataloader, device='cuda', 
                     is_snn=False, model_params={}, snn_type=None):
    """
    Calculate metrics for both CNN and SNN models with NVIDIA SMI energy measurement.
    
    Args:
        model_class: The model class to instantiate
        weight_path: Path to the saved model weights
        test_dataloader: DataLoader for test dataset
        device: Device to run the model on ('cuda' or 'cpu')
        is_snn: Boolean flag indicating if the model is an SNN
        model_params: Additional parameters for model instantiation
        snn_type: String indicating SNN architecture type ('basic' or 'classifier')
        
    Returns:
        Dictionary of calculated metrics
    """
    # Load model and weights
    saved_info = torch.load(weight_path, map_location=device, weights_only=False)
    
    # Handle different state dict keys
    if 'model_state_dict' in saved_info:
        model_state = saved_info['model_state_dict']
    else:
        model_state = saved_info  # Assume the entire file is the state dict
    
    model = model_class(**model_params)
    model.load_state_dict(model_state)
    model.to(device)
    model.eval()

    # 1. Parameter Count
    num_params = sum(p.numel() for p in model.parameters())

    # 2. FLOPs/SynOps Calculation
    flops = 0
    synops = 0
    
    if is_snn:
        # SNN-specific calculation
        with torch.no_grad():
            for inputs, _ in tqdm(test_dataloader, desc="Calculating SynOps"):
                inputs = inputs.to(device)
                utils.reset(model)
                
                # Forward pass - handle different return types
                if snn_type == 'basic':
                    # BasicSNN architecture returns (spk_rec, mem_rec)
                    _, _ = model(inputs)
                else:
                    # SNNClassifier architecture returns output directly
                    _ = model(inputs)
                
                # Calculate SynOps by counting spikes from all layers
                batch_synops = 0
                
                # Common attributes to check for both architectures
                for attr in ['spk1', 'spk2', 'spk3']:
                    if hasattr(model, attr) and getattr(model, attr) is not None:
                        batch_synops += getattr(model, attr).sum().item()
                
                synops += batch_synops
        
        # Estimate base FLOPs for non-spiking operations
        inputs, _ = next(iter(test_dataloader))
        inputs = inputs.to(device)
        flops, _ = thop.profile(model, inputs=(inputs,), verbose=False)
        
    else:
        # CNN FLOPs calculation
        inputs, _ = next(iter(test_dataloader))
        inputs = inputs.to(device)
        flops, _ = thop.profile(model, inputs=(inputs,), verbose=False)

    # 3. Execution Speed
    total_samples = 0
    start_time = time.time()
    
    with torch.no_grad():
        for inputs, _ in tqdm(test_dataloader, desc="Measuring execution speed"):
            if is_snn:
                utils.reset(model)

            batch_size = inputs.size(0)
            total_samples += batch_size
            
            inputs = inputs.to(device)
            
            if snn_type == 'basic':
                _, _ = model(inputs)
            else:
                _ = model(inputs)
                
            if device == 'cuda':
                torch.cuda.synchronize()
    
    exec_time = time.time() - start_time
    exec_speed = total_samples / exec_time  # samples per second

    # 4. Energy Consumption using NVIDIA SMI with improved measurement
    def get_power():
        """Get current GPU power draw using nvidia-smi"""
        try:
            result = subprocess.run(['nvidia-smi', '--query-gpu=power.draw', '--format=csv,noheader,nounits'],
                                   stdout=subprocess.PIPE, timeout=2)
            match = re.search(r'\d+\.\d+', result.stdout.decode())
            return float(match.group()) if match else 0.0
        except (subprocess.TimeoutExpired, subprocess.SubprocessError, ValueError, AttributeError):
            return 0.0
    
    # Skip energy measurement if not on CUDA
    if device != 'cuda':
        energy_per_12k = 0
    else:
        # Power measurement thread setup
        power_readings = []
        thread_running = True
        
        def power_thread():
            while thread_running:
                power_reading = get_power()
                if power_reading > 0:
                    power_readings.append(power_reading)
                time.sleep(0.1)  # Sample every 100ms
        
        # Warm-up
        inputs, _ = next(iter(test_dataloader))
        inputs = inputs.to(device)
        
        if snn_type == 'basic':
            _, _ = model(inputs)
        else:
            _ = model(inputs)
            
        torch.cuda.synchronize()
        
        # Start power measurement thread
        t = threading.Thread(target=power_thread)
        t.start()
        
        # Run measurement
        total_energy_samples = 0
        start_energy = time.time()
        
        with torch.no_grad():
            for inputs, _ in tqdm(test_dataloader, desc="Measuring energy consumption"):
                if is_snn:
                    utils.reset(model)
                batch_size = inputs.size(0)
                total_energy_samples += batch_size
                
                inputs = inputs.to(device)
                
                if snn_type == 'basic':
                    _, _ = model(inputs)
                else:
                    _ = model(inputs)
                    
                torch.cuda.synchronize()
        
        # Stop thread and calculate energy
        thread_running = False
        t.join()
        
        duration = time.time() - start_energy
        
        if len(power_readings) > 0:
            avg_power = sum(power_readings) / len(power_readings)
            energy = avg_power * duration  # Watts * seconds = Joules
            energy_per_sample = energy / total_energy_samples
            energy_per_12k = energy_per_sample * 12000
        else:
            energy_per_12k = 0  # Fallback if power readings fail
    
    # Clean up CUDA memory
    if device == 'cuda':
        del model, inputs
        torch.cuda.empty_cache()

    return {
        'parameters': num_params,
        'flops': flops,
        'synops': synops if is_snn else 0,
        'execution_speed': exec_speed,
        'energy_per_12k': energy_per_12k,
        'raw_data': {
            'total_samples': total_samples,
            'exec_time': exec_time,
            'power_readings': power_readings if device == 'cuda' and 'power_readings' in locals() else []
        }
    }

def detect_snn_type(model):
    """
    Detect the type of SNN architecture
    
    Args:
        model: Instantiated model to analyze
        
    Returns:
        String 'basic' or 'classifier' or None if uncertain
    """
    # Check for BasicSNN characteristics
    if hasattr(model, 'num_steps') and hasattr(model, 'lif1') and hasattr(model, 'lif2'):
        return 'basic'
    
    # Check for SNNClassifier characteristics
    if hasattr(model, 'time_steps') and hasattr(model, 'conv1') and hasattr(model, 'conv2') and hasattr(model, 'encoder'):
        return 'classifier'
    
    # If model has forward method that accepts time_step parameter
    import inspect
    sig = inspect.signature(model.forward)
    params = sig.parameters
    
    if 'time_step' in params or 'num_steps' in params:
        return 'basic'
    
    # Unable to determine clearly
    return None

def run_benchmark(snn_model_class, cnn_model_class, snn_weights, cnn_weights, test_dataloader, 
                 snn_params={}, cnn_params={}, device='cuda', auto_detect_snn=True):
    """
    Run a full benchmark comparison between SNN and CNN models
    
    Args:
        snn_model_class: SNN model class to instantiate
        cnn_model_class: CNN model class to instantiate
        snn_weights: Path to SNN model weights
        cnn_weights: Path to CNN model weights
        test_dataloader: DataLoader for test dataset
        snn_params: Additional parameters for SNN model instantiation
        cnn_params: Additional parameters for CNN model instantiation
        device: Device to run models on ('cuda' or 'cpu')
        auto_detect_snn: Automatically detect SNN architecture type
        
    Returns:
        Tuple of (cnn_metrics, snn_metrics, comparison)
    """
    print(f"Running benchmarks on {device}...")
    
    # Auto-detect SNN type if requested
    snn_type = None
    if auto_detect_snn:
        temp_model = snn_model_class(**snn_params)
        snn_type = detect_snn_type(temp_model)
        print(f"Detected SNN architecture: {snn_type}")
        del temp_model
        if device == 'cuda':
            torch.cuda.empty_cache()
    
    print("\n=== CNN Model ===")
    cnn_metrics = calculate_metrics(
        model_class=cnn_model_class,
        weight_path=cnn_weights,
        test_dataloader=test_dataloader,
        is_snn=False,
        model_params=cnn_params,
        device=device
    )
    
    print("\n=== SNN Model ===")
    snn_metrics = calculate_metrics(
        model_class=snn_model_class,
        weight_path=snn_weights,
        test_dataloader=test_dataloader,
        is_snn=True,
        model_params=snn_params,
        device=device,
        snn_type=snn_type
    )
    
    comparison = valid_comparison(cnn_metrics, snn_metrics)
    
    # Print detailed report
    print("\n" + "="*50)
    print("BENCHMARK RESULTS")
    print("="*50)
    
    print("\n== CNN METRICS ==")
    print(f"Parameters: {cnn_metrics['parameters']:,}")
    print(f"FLOPs: {cnn_metrics['flops']:,}")
    print(f"Execution speed: {cnn_metrics['execution_speed']:.2f} samples/sec")
    print(f"Energy per 12K samples: {cnn_metrics['energy_per_12k']:.2f} Joules")
    
    print("\n== SNN METRICS ==")
    print(f"Parameters: {snn_metrics['parameters']:,}")
    print(f"FLOPs: {snn_metrics['flops']:,}")
    print(f"SynOps: {snn_metrics['synops']:,}")
    print(f"Execution speed: {snn_metrics['execution_speed']:.2f} samples/sec")
    print(f"Energy per 12K samples: {snn_metrics['energy_per_12k']:.2f} Joules")
    
    print("\n== COMPARISON ==")
    print(f"Parameter ratio (SNN/CNN): {comparison['parameter_ratio']:.2f}x")
    print(f"Effective operations ratio (SynOps/FLOPs): {comparison['operations']['effective_ops_ratio']:.4f}x")
    print(f"Speed ratio (SNN/CNN): {comparison['speed_ratio']:.2f}x")
    print(f"Energy ratio (SNN/CNN): {comparison['energy']['energy_ratio']:.2f}x")
    print(f"Energy efficiency ratio: {comparison['energy']['energy_efficiency_ratio']:.2f}x")
    
    return cnn_metrics, snn_metrics, comparison

def valid_comparison(cnn_metrics, snn_metrics):
    """
    Generate meaningful comparison between SNN and CNN metrics
    
    Args:
        cnn_metrics: Dictionary of CNN metrics from calculate_metrics
        snn_metrics: Dictionary of SNN metrics from calculate_metrics
        
    Returns:
        Dictionary of comparative metrics
    """
    # Calculate the effective operations ratio (SynOps / FLOPs)
    effective_ops_ratio = snn_metrics['synops'] / cnn_metrics['flops'] if cnn_metrics['flops'] > 0 else 0
    
    # Calculate energy efficiency (energy per operation)
    cnn_energy_per_op = cnn_metrics['energy_per_12k'] / cnn_metrics['flops'] if cnn_metrics['flops'] > 0 else 0
    snn_energy_per_op = snn_metrics['energy_per_12k'] / snn_metrics['synops'] if snn_metrics['synops'] > 0 else 0
    energy_efficiency_ratio = cnn_energy_per_op / snn_energy_per_op if snn_energy_per_op > 0 else 0
    
    return {
        'parameter_ratio': snn_metrics['parameters'] / cnn_metrics['parameters'],
        'operations': {
            'cnn_flops': cnn_metrics['flops'],
            'snn_synops': snn_metrics['synops'],
            'effective_ops_ratio': effective_ops_ratio
        },
        'speed_ratio': snn_metrics['execution_speed'] / cnn_metrics['execution_speed'],
        'energy': {
            'cnn_energy_per_12k': cnn_metrics['energy_per_12k'],
            'snn_energy_per_12k': snn_metrics['energy_per_12k'],
            'energy_ratio': snn_metrics['energy_per_12k'] / cnn_metrics['energy_per_12k'] 
                if cnn_metrics['energy_per_12k'] > 0 else 0,
            'energy_efficiency_ratio': energy_efficiency_ratio
        }
    }

In [9]:
import sys
import os

sys.path.append(os.path.abspath("../.."))

In [14]:
from src.models import BasicSNN, BinaryCNN3, BasicSNNRate, BinaryAbstractionCNN, SNNClassifier

In [11]:
base_dir = "../../models/checkpoints"

available_dirs = {
    0 : "cnn",
    1 : "multi_cnn",
    2 : "basic_snn",
    3 : "basic_multisnn",
    4 : "paper_snn",
    5 : "paper_multisnn"
}

In [18]:
calculate_metrics(BasicSNNRate, f"{base_dir}/basic_snn/model_ratev1bg1.pt", test_loader_spike, device='cuda', is_snn=True, model_params={'num_steps': 4, 'beta': 1})

Calculating SynOps:   0%|          | 0/300 [00:00<?, ?it/s]

Measuring energy consumption: 100%|██████████| 300/300 [00:13<00:00, 21.64it/s]


{'parameters': 2418,
 'flops': 3942400.0,
 'synops': 4466266.0,
 'execution_speed': 927.6004379796761,
 'energy_per_12k': 114.18791474356884,
 'raw_data': {'total_samples': 12000,
  'exec_time': 12.936604499816895,
  'power_readings': [8.17,
   8.17,
   8.16,
   8.16,
   8.17,
   8.17,
   8.15,
   8.17,
   8.15,
   8.19,
   8.19,
   8.19,
   8.21,
   8.2,
   8.2,
   8.2,
   8.21,
   8.19,
   8.23,
   8.23,
   8.25,
   8.25,
   8.25,
   8.23,
   8.21,
   8.21,
   8.2,
   8.17,
   8.18,
   8.18,
   8.16,
   8.18,
   8.15,
   8.19,
   8.21,
   8.23,
   8.22,
   8.22,
   8.24,
   8.19,
   8.15,
   8.17,
   8.13,
   8.14,
   8.15,
   8.14,
   8.18,
   8.22,
   8.21,
   8.2,
   8.2,
   8.2,
   8.15,
   8.14,
   8.18,
   8.17,
   8.2,
   8.19,
   8.2,
   8.18,
   8.18,
   8.18,
   8.17,
   8.2,
   8.2,
   8.22,
   8.21,
   8.21,
   8.18,
   8.22,
   8.23,
   8.21,
   8.25,
   8.24,
   8.23,
   8.19,
   8.17,
   8.16,
   8.12,
   8.1,
   8.09,
   8.08]}}

In [15]:
calculate_metrics(BinaryCNN3, f"{base_dir}/cnn/bcnn3_crossentropy_plateau.pt", test_loader, device='cuda', is_snn=False, model_params={})

Measuring energy consumption: 100%|██████████| 300/300 [00:10<00:00, 29.97it/s]


{'parameters': 138178,
 'flops': 62343680.0,
 'synops': 0,
 'execution_speed': 1393.2619154733763,
 'energy_per_12k': 83.33037986359355,
 'raw_data': {'total_samples': 12000,
  'exec_time': 8.612881660461426,
  'power_readings': [8.24,
   8.28,
   8.28,
   8.32,
   8.31,
   8.31,
   8.33,
   8.36,
   8.34,
   8.37,
   8.38,
   8.39,
   8.4,
   8.39,
   8.42,
   8.4,
   8.33,
   8.32,
   8.32,
   8.3,
   8.3,
   8.27,
   8.28,
   8.27,
   8.27,
   8.25,
   8.26,
   8.29,
   8.3,
   8.3,
   8.31,
   8.31,
   8.35,
   8.37,
   8.34,
   8.3,
   8.3,
   8.26,
   8.2,
   8.19,
   8.18,
   8.16,
   8.2,
   8.17,
   8.19,
   8.16,
   8.17,
   8.16,
   8.14,
   8.16,
   8.16,
   8.16,
   8.15,
   8.18,
   8.2,
   8.18,
   8.22,
   8.24,
   8.28]}}

In [20]:
calculate_metrics(SNNClassifier, f"{base_dir}/paper_snn/modelv1_best.pt", test_loader, device='cuda', is_snn=True, model_params={})

Calculating SynOps:   0%|          | 0/300 [00:00<?, ?it/s]

Measuring energy consumption: 100%|██████████| 300/300 [00:11<00:00, 25.29it/s]


{'parameters': 3382,
 'flops': 7905280.0,
 'synops': 16478643.0,
 'execution_speed': 1089.3682567921176,
 'energy_per_12k': 97.03002070069313,
 'raw_data': {'total_samples': 12000,
  'exec_time': 11.01555871963501,
  'power_readings': [8.17,
   8.13,
   8.14,
   8.16,
   8.11,
   8.13,
   8.12,
   8.16,
   8.14,
   8.14,
   8.14,
   8.15,
   8.14,
   8.12,
   8.14,
   8.14,
   8.15,
   8.18,
   8.15,
   8.15,
   8.12,
   8.1,
   8.15,
   8.16,
   8.16,
   8.16,
   8.16,
   8.16,
   8.11,
   8.09,
   8.1,
   8.09,
   8.1,
   8.1,
   8.13,
   8.13,
   8.12,
   8.12,
   8.14,
   8.16,
   8.15,
   8.18,
   8.18,
   8.22,
   8.22,
   8.2,
   8.2,
   8.22,
   8.16,
   8.12,
   8.13,
   8.12,
   8.1,
   8.11,
   8.1,
   8.14,
   8.12,
   8.17,
   8.19,
   8.17,
   8.19,
   8.18,
   8.18,
   8.14,
   8.14,
   8.16,
   8.14,
   8.11]}}

In [ ]:
import torch
import time
import re
import subprocess
import threading
from tqdm import tqdm
import thop
import snntorch as snn
from typing import Dict, Any, Optional
from torch.utils.data import DataLoader

class Benchmark:
    def __init__(self, device: str = 'cuda', precision: str = 'fp32'):
        self.device = device
        self.precision = precision
        self._register_custom_handlers()

    def _register_custom_handlers(self):
        """Register custom FLOPs handlers for SNN layers"""
        @thop.profile.register(snn.Leaky)
        def profile_lif(layer, inputs, outputs):
            # Membrane update: 5 FLOPS per neuron
            # (v_new = decay * (v - threshold) + input)
            # Spike generation: 1 FLOP (comparison)
            # Reset: 1 FLOP
            flops = inputs[0].numel() * 7
            return flops, 0

        @thop.profile.register(snn.Synaptic)
        def profile_synaptic(layer, inputs, outputs):
            # Synaptic current update: 3 FLOPS per connection
            flops = inputs[0].numel() * 3
            return flops, 0

    def _count_synops(self, model: torch.nn.Module) -> int:
        """Recursively count spikes (SynOps) in all spiking layers"""
        synops = 0
        for module in model.modules():
            if isinstance(module, (snn.Leaky, snn.Synaptic)):
                if hasattr(module, 'spk') and module.spk is not None:
                    synops += module.spk.sum().item()
        return synops

    def _measure_power(self, stop_event: threading.Event, power_readings: list):
        """Power measurement thread"""
        while not stop_event.is_set():
            try:
                result = subprocess.run(
                    ['nvidia-smi', '--query-gpu=power.draw', '--format=csv,noheader,nounits'],
                    stdout=subprocess.PIPE, 
                    timeout=2
                )
                match = re.search(r'\d+\.\d+', result.stdout.decode())
                if match:
                    power_readings.append(float(match.group()))
            except:
                pass
            time.sleep(0.1)

    def _prepare_model(self, model_class, weight_path, model_params: Dict[str, Any]):
        """Load and prepare model"""
        model = model_class(**model_params)
        
        # Load weights
        checkpoint = torch.load(weight_path, map_location=self.device)
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            model.load_state_dict(checkpoint)
            
        model = model.to(self.device)
        model.eval()
        
        # Precision handling
        if self.precision == 'fp16':
            model = model.half()
        return model

    def run(self, model_class, weight_path: str, dataloader: DataLoader, 
           is_snn: bool = False, model_params: Dict[str, Any] = {}, 
           timesteps: int = 25) -> Dict[str, float]:
        """Run full benchmark"""
        metrics = {}
        
        # 1. Prepare model
        model = self._prepare_model(model_class, weight_path, model_params)
        
        # 2. Parameter count
        metrics['parameters'] = sum(p.numel() for p in model.parameters())
        
        # 3. FLOPs calculation
        inputs, _ = next(iter(dataloader))
        inputs = inputs.to(self.device)
        if self.precision == 'fp16':
            inputs = inputs.half()
            
        flops, _ = thop.profile(model, inputs=(inputs,), verbose=False)
        metrics['flops'] = flops * (timesteps if is_snn else 1)
        
        # 4. Execution speed and SynOps
        synops = 0
        total_samples = 0
        start_time = time.time()
        
        with torch.no_grad():
            for inputs, _ in tqdm(dataloader, desc="Processing"):
                inputs = inputs.to(self.device)
                if self.precision == 'fp16':
                    inputs = inputs.half()
                
                # SNN specific processing
                if is_snn:
                    # Reset membrane potentials
                    utils.reset(model)
                    
                    # Temporal simulation
                    for _ in range(timesteps):
                        _ = model(inputs)
                        synops += self._count_synops(model)
                else:
                    _ = model(inputs)
                
                total_samples += inputs.size(0)
                
                if self.device == 'cuda':
                    torch.cuda.synchronize()
        
        metrics['execution_speed'] = total_samples / (time.time() - start_time)
        metrics['synops'] = synops if is_snn else 0
        
        # 5. Energy consumption
        metrics['energy'] = 0.0
        if self.device == 'cuda':
            power_readings = []
            stop_event = threading.Event()
            power_thread = threading.Thread(
                target=self._measure_power,
                args=(stop_event, power_readings)
            
            # Warmup
            _ = model(inputs)
            torch.cuda.synchronize()
            
            # Measurement
            power_thread.start()
            start_energy = time.time()
            
            with torch.no_grad():
                for inputs, _ in dataloader:
                    inputs = inputs.to(self.device)
                    if is_snn:
                        for _ in range(timesteps):
                            _ = model(inputs)
                    else:
                        _ = model(inputs)
                    torch.cuda.synchronize()
            
            duration = time.time() - start_energy
            stop_event.set()
            power_thread.join()
            
            if power_readings:
                avg_power = sum(power_readings) / len(power_readings)
                metrics['energy'] = (avg_power * duration) / total_samples * 10000  # J/10k samples

        # Cleanup
        del model
        torch.cuda.empty_cache()
        
        return metrics

    def compare(self, cnn_metrics: Dict[str, float], snn_metrics: Dict[str, float]) -> Dict[str, float]:
        """Generate meaningful comparison between SNN and CNN"""
        return {
            'parameter_ratio': snn_metrics['parameters'] / cnn_metrics['parameters'],
            'flops_ratio': snn_metrics['flops'] / cnn_metrics['flops'],
            'energy_ratio': snn_metrics['energy'] / cnn_metrics['energy'],
            'effective_ops_ratio': (
                snn_metrics['synops'] + snn_metrics['flops']) / cnn_metrics['flops'],
            'efficiency_ratio': (
                cnn_metrics['flops'] / cnn_metrics['energy']) / 
                (snn_metrics['synops'] / snn_metrics['energy'])
        }

In [ ]:
run_benchmark(SNNClassifier, BinaryCNN3, f"{base_dir}/paper_snn/modelv1_best.pt", f"{base_dir}/cnn/bcnn3_crossentropy_plateau.pt", test_loader, device='cuda', auto_detect_snn=True)

Running benchmarks on cuda...
Detected SNN architecture: classifier

=== CNN Model ===


Measuring energy consumption: 100%|██████████| 300/300 [00:13<00:00, 22.74it/s]



=== SNN Model ===


Measuring execution speed:  98%|█████████▊| 293/300 [00:13<00:00, 21.55it/s]